# 11 · Análisis descriptivo independiente de viajes · enero-febrero de 2023

Este notebook estudia exclusivamente los **viajes observados de BiciMAD** entre enero y febrero de 2023. Su finalidad es descriptiva: caracterizar cobertura, volumen, patrones temporales, duración, actividad por estación y principales pares origen-destino.

> **Límite metodológico:** no existen estados horarios de las estaciones para enero-febrero de 2023. Por tanto, este periodo no se utiliza para crear etiquetas de vaciado/saturación, evaluar el modelo, recalibrar probabilidades ni generar recomendaciones operativas.

## Separación respecto al flujo predictivo

- Las entradas son los CSV mensuales de **viajes**, no las tablas estación-hora ni los artefactos de los notebooks 07-10.
- Los nombres históricos de estaciones se recuperan de los mismos ficheros originales de viajes; no se emplean estados de estación.
- `arrivals - departures` representa únicamente el balance de viajes observados. No equivale a disponibilidad, inventario, vaciado ni saturación.
- Las comparaciones mensuales respetan la cobertura: febrero termina el día 18 y no se tratará como un mes completo.
- Las asociaciones temporales son descriptivas y no deben interpretarse causalmente.

## Entorno, rutas y parámetros de calidad

In [ ]:
from pathlib import Path
import calendar
import json
import warnings

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')
plt.style.use('seaborn-v0_8-whitegrid')

def locate_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'notebooks').exists() and (candidate / 'Bases de datos').exists():
            return candidate
    raise FileNotFoundError('No se ha localizado la raíz del proyecto TFM - Xabi.')

PROJECT_ROOT = locate_project_root()
TRIPS_DIR = PROJECT_ROOT / 'notebooks' / 'Datos analiticos' / 'BiciMAD' / 'viajes'
OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'Datos analiticos' / 'analisis_descriptivo_viajes_2023_enero_febrero'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_PERIOD = pd.Timestamp('2023-01-01 00:00:00')
END_PERIOD = pd.Timestamp('2023-03-01 00:00:00')
MIN_DURATION_SECONDS = 60
MAX_DURATION_SECONDS = 120 * 60
TOP_STATIONS = 15
TOP_OD_PAIRS = 20

print('Raíz:', PROJECT_ROOT)
print('Entradas:', TRIPS_DIR)
print('Salidas:', OUTPUT_DIR)

## Carga de viajes y referencia histórica de estaciones

Las particiones analíticas eliminan las filas vacías intercaladas en los originales y unifican el esquema. Los originales se leen únicamente para obtener el nombre y la dirección más frecuentes asociados a cada identificador de estación durante el propio periodo.

In [ ]:
trip_paths = {
    '2023-01': TRIPS_DIR / 'viajes_202301.csv',
    '2023-02': TRIPS_DIR / 'viajes_202302.csv',
}
raw_paths = {
    '2023-01': next(iter((PROJECT_ROOT / 'Bases de datos').rglob('trips_23_01_January.csv')), None),
    '2023-02': next(iter((PROJECT_ROOT / 'Bases de datos').rglob('trips_23_02_February.csv')), None),
}

missing_inputs = [str(path) for path in trip_paths.values() if not path.exists()]
missing_raw = [month for month, path in raw_paths.items() if path is None or not path.exists()]
assert not missing_inputs, f'Faltan particiones analíticas: {missing_inputs}'
assert not missing_raw, f'Faltan originales para recuperar nombres: {missing_raw}'

trip_columns = [
    'source_file', 'source_format', 'start_at', 'end_at', 'duration_seconds',
    'origin_station_id', 'destination_station_id', 'origin_dock_id',
    'destination_dock_id', 'origin_latitude', 'origin_longitude',
    'destination_latitude', 'destination_longitude',
]
trip_parts = []
input_rows_by_file = {}
for month, path in trip_paths.items():
    part = pd.read_csv(path, usecols=trip_columns, low_memory=False)
    part['input_month'] = month
    input_rows_by_file[path.name] = len(part)
    trip_parts.append(part)
trips_raw = pd.concat(trip_parts, ignore_index=True)

def most_frequent_nonempty(values: pd.Series):
    cleaned = values.dropna().astype(str).str.strip()
    cleaned = cleaned[cleaned.ne('')]
    return cleaned.value_counts().index[0] if not cleaned.empty else pd.NA

station_label_parts = []
raw_name_columns = [
    'station_unlock', 'unlock_station_name', 'address_unlock',
    'station_lock', 'lock_station_name', 'address_lock',
]
for month, path in raw_paths.items():
    raw_names = pd.read_csv(path, sep=';', usecols=raw_name_columns, low_memory=False)
    for side in ('unlock', 'lock'):
        station_label_parts.append(
            raw_names[[f'station_{side}', f'{side}_station_name', f'address_{side}']]
            .rename(columns={
                f'station_{side}': 'station_id',
                f'{side}_station_name': 'station_name',
                f'address_{side}': 'address',
            })
            .dropna(subset=['station_id'])
        )
station_labels_long = pd.concat(station_label_parts, ignore_index=True)
station_labels_long['station_id'] = pd.to_numeric(station_labels_long['station_id'], errors='coerce').astype('Int64')
station_reference = (
    station_labels_long.dropna(subset=['station_id'])
    .groupby('station_id', as_index=False)
    .agg(station_name=('station_name', most_frequent_nonempty), address=('address', most_frequent_nonempty))
)

print('Filas analíticas cargadas:', f'{len(trips_raw):,}')
print('Filas por fichero:', input_rows_by_file)
print('Estaciones con etiqueta histórica:', len(station_reference))

## Normalización mínima y auditoría de calidad

No se eliminan registros para calcular volumen salvo que la fecha de inicio sea inválida o quede fuera del periodo. La ventana de 1-120 minutos se utiliza **solo** para describir duraciones típicas, evitando que incidencias extremas dominen las estadísticas.

In [ ]:
trips = trips_raw.copy()
for column in ['start_at', 'end_at']:
    trips[column] = pd.to_datetime(trips[column], errors='coerce')
for column in ['duration_seconds', 'origin_latitude', 'origin_longitude', 'destination_latitude', 'destination_longitude']:
    trips[column] = pd.to_numeric(trips[column], errors='coerce')
for column in ['origin_station_id', 'destination_station_id', 'origin_dock_id', 'destination_dock_id']:
    trips[column] = pd.to_numeric(trips[column], errors='coerce').astype('Int64')

invalid_start_rows = int(trips['start_at'].isna().sum())
out_of_period_rows = int((trips['start_at'].notna() & ~trips['start_at'].between(START_PERIOD, END_PERIOD, inclusive='left')).sum())
trips = trips[trips['start_at'].between(START_PERIOD, END_PERIOD, inclusive='left')].copy()

trips['month'] = trips['start_at'].dt.to_period('M').astype(str)
trips['date'] = trips['start_at'].dt.normalize()
trips['hour'] = trips['start_at'].dt.hour
trips['start_hour'] = trips['start_at'].dt.floor('h')
trips['weekday_number'] = trips['start_at'].dt.weekday
weekday_names = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo']
trips['weekday'] = pd.Categorical(
    trips['weekday_number'].map(dict(enumerate(weekday_names))),
    categories=weekday_names, ordered=True,
)
trips['week_segment'] = np.where(trips['weekday_number'] >= 5, 'fin_de_semana', 'lunes_a_viernes')
trips['is_station_to_station'] = trips['origin_station_id'].notna() & trips['destination_station_id'].notna()
trips['is_same_station'] = trips['is_station_to_station'] & trips['origin_station_id'].eq(trips['destination_station_id'])
trips['duration_for_analysis'] = trips['duration_seconds'].where(
    trips['duration_seconds'].between(MIN_DURATION_SECONDS, MAX_DURATION_SECONDS, inclusive='both')
)

signature_columns = [
    'start_at', 'end_at', 'duration_seconds', 'origin_station_id',
    'destination_station_id', 'origin_dock_id', 'destination_dock_id',
]
quality_audit = pd.DataFrame([
    {'control': 'input_rows', 'value': len(trips_raw), 'interpretation': 'filas de las dos particiones normalizadas'},
    {'control': 'valid_period_rows', 'value': len(trips), 'interpretation': 'viajes con inicio válido dentro de enero-febrero de 2023'},
    {'control': 'invalid_start_rows', 'value': invalid_start_rows, 'interpretation': 'fecha de inicio no interpretable'},
    {'control': 'out_of_period_rows', 'value': out_of_period_rows, 'interpretation': 'inicio fuera del intervalo declarado'},
    {'control': 'missing_end_rows', 'value': int(trips['end_at'].isna().sum()), 'interpretation': 'sin fecha final'},
    {'control': 'missing_duration_rows', 'value': int(trips['duration_seconds'].isna().sum()), 'interpretation': 'sin duración informada'},
    {'control': 'duration_below_1_minute', 'value': int(trips['duration_seconds'].lt(MIN_DURATION_SECONDS).sum()), 'interpretation': 'se conservan para volumen, no para duración típica'},
    {'control': 'duration_above_120_minutes', 'value': int(trips['duration_seconds'].gt(MAX_DURATION_SECONDS).sum()), 'interpretation': 'se conservan para volumen, no para duración típica'},
    {'control': 'non_positive_duration_rows', 'value': int(trips['duration_seconds'].le(0).sum()), 'interpretation': 'incidencias de calidad'},
    {'control': 'station_to_station_rows', 'value': int(trips['is_station_to_station'].sum()), 'interpretation': 'soporte del análisis por estación y OD'},
    {'control': 'same_station_rows', 'value': int(trips['is_same_station'].sum()), 'interpretation': 'origen y destino con el mismo identificador'},
    {'control': 'duplicate_signatures', 'value': int(trips.duplicated(signature_columns, keep=False).sum()), 'interpretation': 'filas implicadas; no se eliminan porque no existe trip_id fiable'},
    {'control': 'state_columns_present', 'value': bool({'bikes_available', 'docks_available', 'capacity'} & set(trips.columns)), 'interpretation': 'debe ser False'},
    {'control': 'risk_or_prediction_columns_present', 'value': bool({'prediction', 'risk_class_1h', 'model_action_signal'} & set(trips.columns)), 'interpretation': 'debe ser False'},
])

assert len(trips) == sum(input_rows_by_file.values())
assert trips['start_at'].min() >= START_PERIOD and trips['start_at'].max() < END_PERIOD
assert not bool(quality_audit.loc[quality_audit.control == 'state_columns_present', 'value'].iloc[0])
assert not bool(quality_audit.loc[quality_audit.control == 'risk_or_prediction_columns_present', 'value'].iloc[0])
display(quality_audit)

## Cobertura temporal

Se distingue entre días observados y días completos. Un día se considera completo para las comparaciones intradía cuando existen viajes tanto en su primera como en su última hora. Es una regla de cobertura, no una selección basada en resultados.

In [ ]:
observed_daily = (
    trips.groupby('date', as_index=False)
    .agg(
        trips=('start_at', 'size'),
        first_trip=('start_at', 'min'),
        last_trip=('start_at', 'max'),
        observed_hours=('start_hour', 'nunique'),
        station_to_station_trips=('is_station_to_station', 'sum'),
        same_station_trips=('is_same_station', 'sum'),
    )
)
observed_daily['complete_day'] = (
    observed_daily['first_trip'].lt(observed_daily['date'] + pd.Timedelta(hours=1))
    & observed_daily['last_trip'].ge(observed_daily['date'] + pd.Timedelta(hours=23))
)

expected_dates = pd.DataFrame({'date': pd.date_range(START_PERIOD, END_PERIOD - pd.Timedelta(days=1), freq='D')})
daily_coverage = expected_dates.merge(observed_daily, on='date', how='left')
daily_coverage['trips'] = daily_coverage['trips'].fillna(0).astype(int)
daily_coverage['observed_hours'] = daily_coverage['observed_hours'].fillna(0).astype(int)
daily_coverage['station_to_station_trips'] = daily_coverage['station_to_station_trips'].fillna(0).astype(int)
daily_coverage['same_station_trips'] = daily_coverage['same_station_trips'].fillna(0).astype(int)
daily_coverage['complete_day'] = daily_coverage['complete_day'].fillna(False).astype(bool)
daily_coverage['observed_day'] = daily_coverage['trips'].gt(0)
daily_coverage['month'] = daily_coverage['date'].dt.to_period('M').astype(str)
daily_coverage['weekday_number'] = daily_coverage['date'].dt.weekday
daily_coverage['weekday'] = pd.Categorical(
    daily_coverage['weekday_number'].map(dict(enumerate(weekday_names))),
    categories=weekday_names, ordered=True,
)

month_coverage = (
    daily_coverage.groupby('month', as_index=False, observed=True)
    .agg(
        expected_days=('date', 'size'),
        observed_days=('observed_day', 'sum'),
        complete_days=('complete_day', 'sum'),
        first_trip=('first_trip', 'min'),
        last_trip=('last_trip', 'max'),
    )
)
month_coverage['month_complete'] = month_coverage['observed_days'].eq(month_coverage['expected_days']) & month_coverage['complete_days'].eq(month_coverage['expected_days'])

display(month_coverage)
assert bool(month_coverage.loc[month_coverage.month == '2023-01', 'month_complete'].iloc[0])
assert not bool(month_coverage.loc[month_coverage.month == '2023-02', 'month_complete'].iloc[0])

## Volumen diario y comparación mensual ajustada por cobertura

El total mensual de febrero se conserva como dato observado, pero no se compara directamente con enero. La comparación interpretable utiliza viajes medios por **día completo**.

In [ ]:
complete_dates = set(daily_coverage.loc[daily_coverage['complete_day'], 'date'])
trips['complete_day'] = trips['date'].isin(complete_dates)

monthly_counts = (
    trips.groupby('month', as_index=False)
    .agg(
        observed_trips=('start_at', 'size'),
        station_to_station_trips=('is_station_to_station', 'sum'),
        same_station_trips=('is_same_station', 'sum'),
        unique_origin_stations=('origin_station_id', 'nunique'),
        unique_destination_stations=('destination_station_id', 'nunique'),
    )
)
complete_daily = daily_coverage[daily_coverage['complete_day']].copy()
complete_day_stats = (
    complete_daily.groupby('month', as_index=False, observed=True)
    .agg(
        complete_days=('date', 'size'),
        mean_trips_per_complete_day=('trips', 'mean'),
        median_trips_per_complete_day=('trips', 'median'),
        min_trips_complete_day=('trips', 'min'),
        max_trips_complete_day=('trips', 'max'),
    )
)
monthly_summary = monthly_counts.merge(complete_day_stats, on='month', how='left').merge(
    month_coverage[['month', 'expected_days', 'observed_days', 'month_complete']], on='month', how='left'
)
monthly_summary['station_to_station_share'] = monthly_summary['station_to_station_trips'] / monthly_summary['observed_trips']
monthly_summary['same_station_share_within_station_trips'] = monthly_summary['same_station_trips'] / monthly_summary['station_to_station_trips']
display(monthly_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(daily_coverage['date'], daily_coverage['trips'], color='#1f77b4', linewidth=1.5)
axes[0].scatter(
    daily_coverage.loc[~daily_coverage['complete_day'], 'date'],
    daily_coverage.loc[~daily_coverage['complete_day'], 'trips'],
    color='#c44e52', s=24, label='día parcial o sin cobertura', zorder=3,
)
axes[0].axvline(trips['start_at'].max(), color='#c44e52', linestyle='--', alpha=0.8, label='último viaje observado')
axes[0].set(title='Viajes observados por día', xlabel='Fecha', ylabel='Viajes')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=35)

axes[1].bar(complete_day_stats['month'], complete_day_stats['mean_trips_per_complete_day'], color=['#4c78a8', '#f58518'])
for index, row in complete_day_stats.iterrows():
    axes[1].text(index, row['mean_trips_per_complete_day'], f"{row['mean_trips_per_complete_day']:,.0f}", ha='center', va='bottom')
axes[1].set(title='Media diaria usando únicamente días completos', xlabel='Mes', ylabel='Viajes por día completo')
fig.suptitle('Cobertura y volumen de viajes · enero-febrero de 2023', fontsize=14)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'viajes_diarios_y_cobertura.png', dpi=160, bbox_inches='tight')
plt.show()

## Patrones horarios y semanales

Se utilizan días completos para que el corte de febrero no reduzca artificialmente determinadas horas. `lunes_a_viernes` no significa día laborable oficial: no se incorpora un calendario de festivos de 2023.

In [ ]:
complete_trips = trips[trips['complete_day']].copy()
complete_days_by_month = complete_daily.groupby('month', observed=True)['date'].nunique()
hourly_summary = (
    complete_trips.groupby(['month', 'hour'], as_index=False)
    .size().rename(columns={'size': 'trips'})
)
hourly_summary['complete_days'] = hourly_summary['month'].map(complete_days_by_month)
hourly_summary['mean_trips_per_complete_day'] = hourly_summary['trips'] / hourly_summary['complete_days']

weekday_daily = complete_daily.groupby(['month', 'weekday'], as_index=False, observed=True)['trips'].agg(['sum', 'mean', 'median', 'count']).reset_index()
weekday_daily = weekday_daily.rename(columns={'sum': 'trips', 'mean': 'mean_trips_per_day', 'median': 'median_trips_per_day', 'count': 'days'})

week_segment_summary = (
    complete_daily.assign(week_segment=np.where(complete_daily['weekday_number'] >= 5, 'fin_de_semana', 'lunes_a_viernes'))
    .groupby(['month', 'week_segment'], as_index=False, observed=True)
    .agg(days=('date', 'size'), mean_trips_per_day=('trips', 'mean'), median_trips_per_day=('trips', 'median'))
)

display(hourly_summary.sort_values(['month', 'mean_trips_per_complete_day'], ascending=[True, False]).groupby('month').head(5))
display(week_segment_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = {'2023-01': '#4c78a8', '2023-02': '#f58518'}
for month, frame in hourly_summary.groupby('month'):
    axes[0].plot(frame['hour'], frame['mean_trips_per_complete_day'], marker='o', label=month, color=colors[month])
axes[0].set(title='Perfil horario medio por día completo', xlabel='Hora', ylabel='Viajes medios')
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend()

x = np.arange(len(weekday_names))
width = 0.38
for offset, month in zip([-width / 2, width / 2], ['2023-01', '2023-02']):
    frame = weekday_daily[weekday_daily['month'].eq(month)].set_index('weekday').reindex(weekday_names)
    axes[1].bar(x + offset, frame['mean_trips_per_day'], width, label=month, color=colors[month])
axes[1].set(title='Media diaria por día de la semana', xlabel='Día', ylabel='Viajes medios')
axes[1].set_xticks(x, weekday_names, rotation=35, ha='right')
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'patrones_horarios_y_semanales.png', dpi=160, bbox_inches='tight')
plt.show()

## Duración de los viajes

La duración procede del campo informado por BiciMAD. Las métricas centrales se calculan sobre 1-120 minutos y se acompañan del soporte excluido; este filtro no altera los recuentos de viajes de las secciones anteriores.

In [ ]:
duration_valid = trips.dropna(subset=['duration_for_analysis']).copy()
duration_valid['duration_minutes'] = duration_valid['duration_for_analysis'] / 60
duration_summary = (
    duration_valid.groupby('month')['duration_minutes']
    .agg(
        valid_duration_trips='size',
        mean_minutes='mean',
        median_minutes='median',
        p25_minutes=lambda values: values.quantile(0.25),
        p75_minutes=lambda values: values.quantile(0.75),
        p95_minutes=lambda values: values.quantile(0.95),
    ).reset_index()
)
duration_support = trips.groupby('month', as_index=False).agg(
    observed_trips=('start_at', 'size'),
    below_1_minute=('duration_seconds', lambda values: values.lt(MIN_DURATION_SECONDS).sum()),
    above_120_minutes=('duration_seconds', lambda values: values.gt(MAX_DURATION_SECONDS).sum()),
)
duration_summary = duration_summary.merge(duration_support, on='month', how='left')
duration_summary['valid_duration_share'] = duration_summary['valid_duration_trips'] / duration_summary['observed_trips']
display(duration_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
bins = np.arange(0, 61, 2)
for month, frame in duration_valid[duration_valid['duration_minutes'].le(60)].groupby('month'):
    axes[0].hist(frame['duration_minutes'], bins=bins, density=True, alpha=0.55, label=month, color=colors[month])
axes[0].set(title='Distribución de duración (detalle 1-60 min)', xlabel='Minutos', ylabel='Densidad')
axes[0].legend()

duration_by_route_type = (
    duration_valid.assign(route_type=np.where(duration_valid['is_same_station'], 'misma estación', 'estaciones distintas'))
    .groupby(['month', 'route_type'], as_index=False)['duration_minutes'].median()
)
route_types = ['estaciones distintas', 'misma estación']
x = np.arange(len(route_types))
for offset, month in zip([-width / 2, width / 2], ['2023-01', '2023-02']):
    frame = duration_by_route_type[duration_by_route_type['month'].eq(month)].set_index('route_type').reindex(route_types)
    axes[1].bar(x + offset, frame['duration_minutes'], width, label=month, color=colors[month])
axes[1].set(title='Mediana de duración por tipo de trayecto', ylabel='Minutos')
axes[1].set_xticks(x, route_types, rotation=15)
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'duracion_viajes.png', dpi=160, bbox_inches='tight')
plt.show()

## Actividad por estación y pares origen-destino

Estas tablas utilizan únicamente viajes con identificador de estación en ambos extremos. La actividad y el balance neto describen movimientos registrados, no estados ni capacidad disponible. Los rankings incluyen siempre el soporte (`departures`, `arrivals` y `total_activity`).

In [ ]:
station_trips = trips[trips['is_station_to_station']].copy()
departures = station_trips.groupby('origin_station_id').size().rename('departures')
arrivals = station_trips.groupby('destination_station_id').size().rename('arrivals')
station_activity = pd.concat([departures, arrivals], axis=1).fillna(0).astype(int).reset_index().rename(columns={'index': 'station_id'})
station_activity['total_activity'] = station_activity['departures'] + station_activity['arrivals']
station_activity['net_arrivals'] = station_activity['arrivals'] - station_activity['departures']
station_activity['absolute_net_flow'] = station_activity['net_arrivals'].abs()
station_activity['departure_share'] = station_activity['departures'] / station_activity['total_activity']
station_activity = station_activity.merge(station_reference, on='station_id', how='left')
station_activity['station_name'] = station_activity['station_name'].fillna('estación ' + station_activity['station_id'].astype(str))

origin_coords = station_trips[['origin_station_id', 'origin_latitude', 'origin_longitude']].rename(
    columns={'origin_station_id': 'station_id', 'origin_latitude': 'latitude', 'origin_longitude': 'longitude'}
)
destination_coords = station_trips[['destination_station_id', 'destination_latitude', 'destination_longitude']].rename(
    columns={'destination_station_id': 'station_id', 'destination_latitude': 'latitude', 'destination_longitude': 'longitude'}
)
station_coordinates = (
    pd.concat([origin_coords, destination_coords], ignore_index=True)
    .dropna(subset=['station_id', 'latitude', 'longitude'])
    .groupby('station_id', as_index=False)[['latitude', 'longitude']].median()
)
station_activity = station_activity.merge(station_coordinates, on='station_id', how='left')

od_pairs = (
    station_trips.groupby(['origin_station_id', 'destination_station_id'], as_index=False)
    .size().rename(columns={'size': 'trips'})
)
origin_names = station_reference[['station_id', 'station_name']].rename(columns={'station_id': 'origin_station_id', 'station_name': 'origin_station_name'})
destination_names = station_reference[['station_id', 'station_name']].rename(columns={'station_id': 'destination_station_id', 'station_name': 'destination_station_name'})
od_pairs = od_pairs.merge(origin_names, on='origin_station_id', how='left').merge(destination_names, on='destination_station_id', how='left')
od_pairs['origin_station_name'] = od_pairs['origin_station_name'].fillna('estación ' + od_pairs['origin_station_id'].astype(str))
od_pairs['destination_station_name'] = od_pairs['destination_station_name'].fillna('estación ' + od_pairs['destination_station_id'].astype(str))
od_pairs['same_station'] = od_pairs['origin_station_id'].eq(od_pairs['destination_station_id'])
od_pairs['share_of_station_trips'] = od_pairs['trips'] / len(station_trips)
od_pairs['route'] = od_pairs['origin_station_name'] + ' → ' + od_pairs['destination_station_name']
od_pairs = od_pairs.sort_values('trips', ascending=False).reset_index(drop=True)
od_pairs['rank_all_pairs'] = np.arange(1, len(od_pairs) + 1)

print('Estaciones con actividad:', len(station_activity))
print('Pares OD distintos:', len(od_pairs))
display(station_activity.nlargest(TOP_STATIONS, 'total_activity')[['station_id', 'station_name', 'departures', 'arrivals', 'total_activity', 'net_arrivals']])
display(od_pairs.loc[~od_pairs['same_station']].head(TOP_OD_PAIRS)[['origin_station_name', 'destination_station_name', 'trips', 'share_of_station_trips']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
top_departures = station_activity.nlargest(TOP_STATIONS, 'departures').sort_values('departures')
top_arrivals = station_activity.nlargest(TOP_STATIONS, 'arrivals').sort_values('arrivals')
axes[0].barh(top_departures['station_name'], top_departures['departures'], color='#4c78a8')
axes[0].set(title='Estaciones con más salidas observadas', xlabel='Salidas')
axes[1].barh(top_arrivals['station_name'], top_arrivals['arrivals'], color='#f58518')
axes[1].set(title='Estaciones con más llegadas observadas', xlabel='Llegadas')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'actividad_estaciones.png', dpi=160, bbox_inches='tight')
plt.show()

spatial = station_activity.dropna(subset=['latitude', 'longitude']).copy()
limit = max(float(spatial['absolute_net_flow'].quantile(0.98)), 1.0)
norm = TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit)
fig, ax = plt.subplots(figsize=(9, 8))
scatter = ax.scatter(
    spatial['longitude'], spatial['latitude'],
    s=np.sqrt(spatial['total_activity']).clip(lower=5) * 2.2,
    c=spatial['net_arrivals'].clip(-limit, limit), cmap='coolwarm', norm=norm,
    alpha=0.75, edgecolor='white', linewidth=0.4,
)
for _, row in spatial.nlargest(10, 'absolute_net_flow').iterrows():
    ax.annotate(str(row['station_name']), (row['longitude'], row['latitude']), xytext=(3, 3), textcoords='offset points', fontsize=7)
fig.colorbar(scatter, ax=ax, label='Llegadas − salidas observadas')
ax.set(title='Balance espacial de viajes observados', xlabel='Longitud', ylabel='Latitud')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'balance_espacial_viajes_observados.png', dpi=160, bbox_inches='tight')
plt.show()

top_od = od_pairs.loc[~od_pairs['same_station']].head(TOP_OD_PAIRS).sort_values('trips')
fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(top_od['route'], top_od['trips'], color='#54a24b')
ax.set(title='Principales pares origen-destino entre estaciones distintas', xlabel='Viajes observados')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'principales_pares_od.png', dpi=160, bbox_inches='tight')
plt.show()

## Exportación, manifiesto y controles finales

Las tablas exportadas pueden utilizarse en la memoria y, de forma opcional, como contexto descriptivo en la futura aplicación. No deben mezclarse con las predicciones o recomendaciones de noviembre-diciembre de 2022.

In [ ]:
export_tables = {
    'auditoria_calidad_viajes_2023.csv': quality_audit,
    'cobertura_mensual_2023.csv': month_coverage,
    'resumen_mensual_viajes_2023.csv': monthly_summary,
    'viajes_por_dia_2023.csv': daily_coverage,
    'viajes_por_hora_2023.csv': hourly_summary,
    'viajes_por_dia_semana_2023.csv': weekday_daily,
    'viajes_por_segmento_semana_2023.csv': week_segment_summary,
    'resumen_duracion_viajes_2023.csv': duration_summary,
    'actividad_por_estacion_2023.csv': station_activity.sort_values('total_activity', ascending=False),
    'pares_origen_destino_2023.csv': od_pairs,
}
for filename, frame in export_tables.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)

final_audit = pd.DataFrame([
    {'control': 'analysis_scope', 'value': 'viajes_descriptivos_2023_enero_febrero'},
    {'control': 'first_trip', 'value': trips['start_at'].min().isoformat()},
    {'control': 'last_trip', 'value': trips['start_at'].max().isoformat()},
    {'control': 'observed_trips', 'value': int(len(trips))},
    {'control': 'observed_days', 'value': int(daily_coverage['observed_day'].sum())},
    {'control': 'complete_days', 'value': int(daily_coverage['complete_day'].sum())},
    {'control': 'february_complete', 'value': bool(month_coverage.loc[month_coverage.month == '2023-02', 'month_complete'].iloc[0])},
    {'control': 'station_state_data_used', 'value': False},
    {'control': 'risk_labels_created', 'value': False},
    {'control': 'model_predictions_used', 'value': False},
    {'control': 'model_evaluation_performed', 'value': False},
    {'control': 'probabilities_or_thresholds_adjusted', 'value': False},
    {'control': 'operational_recommendations_generated', 'value': False},
    {'control': 'net_flow_interpretation', 'value': 'balance_de_viajes_no_disponibilidad'},
])
final_audit.to_csv(OUTPUT_DIR / 'auditoria_alcance_descriptivo_2023.csv', index=False)

manifest = {
    'notebook': '11_analisis_descriptivo_viajes_enero_febrero_2023.ipynb',
    'period_requested': ['2023-01-01', '2023-02-28'],
    'period_observed': [trips['start_at'].min().isoformat(), trips['start_at'].max().isoformat()],
    'input_files': {month: str(path) for month, path in trip_paths.items()},
    'raw_files_used_only_for_station_labels': {month: str(path) for month, path in raw_paths.items()},
    'scope_guards': {
        'station_state_data_used': False,
        'risk_labels_or_predictions_used': False,
        'operational_recommendations_generated': False,
        'february_is_partial': True,
    },
    'duration_analysis_window_seconds': [MIN_DURATION_SECONDS, MAX_DURATION_SECONDS],
    'outputs': sorted([*export_tables.keys(), 'auditoria_alcance_descriptivo_2023.csv', 'viajes_diarios_y_cobertura.png', 'patrones_horarios_y_semanales.png', 'duracion_viajes.png', 'actividad_estaciones.png', 'balance_espacial_viajes_observados.png', 'principales_pares_od.png']),
}
(OUTPUT_DIR / 'manifiesto_analisis_descriptivo_2023.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

assert not final_audit.loc[final_audit.control.isin([
    'station_state_data_used', 'risk_labels_created', 'model_predictions_used',
    'model_evaluation_performed', 'probabilities_or_thresholds_adjusted',
    'operational_recommendations_generated',
]), 'value'].astype(bool).any()

display(final_audit)
print('Resultados guardados en:', OUTPUT_DIR)

## Lectura para el TFM y siguiente paso

Este notebook completa el análisis descriptivo de viajes disponible para 2023 sin atribuirle funciones que los datos no permiten. En la memoria deben separarse claramente:

1. **Modelo y evaluación final:** noviembre-diciembre de 2022, con estados y etiquetas disponibles.
2. **Análisis descriptivo adicional:** viajes observados de enero y parte de febrero de 2023.
3. **Aplicación interactiva:** siguiente fase, mostrando la procedencia temporal de cada componente para no presentar 2023 como validación del modelo.

Limitaciones principales: febrero es parcial; no hay estados horarios; el calendario laboral de 2023 no forma parte de las entradas; los identificadores de viaje no son fiables en el CSV normalizado; y el balance de salidas/llegadas no permite reconstruir inventario.